In [1]:
import os
import sys

import matplotlib
matplotlib.use("Qt5Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pypsa

In [2]:
n = pypsa.Network("networks/WP2024_north-west.nc")

/home/icerydev/HackathonEnv/Code/Hackathons/TPSA_Hackathon/.venv/lib/python3.14/site-packages/pypsa/network/io.py:2082: FutureWarning: pandas infers the `str` dtype for string data since its version 3.0. PyPSA still converts it back to numpy object dtype on import, but will keep it from PyPSA 2.0 on. Set `pypsa.options.api.legacy_string_dtype` explicitly to suppress this warning.
  new_static = _coerce_string_dtypes(new_static)
INFO:pypsa.network.io:Imported network 'TYTFS2024_WP2024_V35 north-west (aggregated)' has buses, carriers, generators, lines, loads


In [10]:
print("Buses:")
print(n.buses)

print("\nGenerators:")
print(n.generators)

print("\nLoads:")
print(n.loads)

print("\nLines:")
print(n.lines)

print("\nExisting storage units (if any):")
print(n.storage_units)

print("\nSnapshots (time steps):")
print(n.snapshots)

Buses:
                  v_nom type         x          y carrier unit location  \
name                                                                      
Ardnagappary      110.0      -8.268818  55.056640      AC                 
Binbane           110.0      -8.263429  54.718937      AC                 
Cathaleen's Fall  110.0      -8.158787  54.495945      AC                 
Clogher           110.0      -7.974266  54.683846      AC                 
Corderry          110.0      -8.195063  54.174663      AC                 
Croaghonagh       110.0      -7.974266  54.683846      AC                 
Drumkeen          110.0      -7.807338  54.865415      AC                 
Glenree           110.0      -8.989324  54.105345      AC                 
Letterkenny       110.0      -7.739387  54.918817      AC                 
Moy               110.0      -9.195397  54.140397      AC                 
Sligo             110.0      -8.527881  54.197868      AC                 
Sorne Hill        

In [6]:
n.plot(bus_sizes=0.0025, margin=0.25)
plt.show()

qt.qpa.wayland: Wayland does not support QWindow::requestActivate()


In [7]:
n.optimize()

/tmp/ipykernel_19769/1261279110.py:1: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n.optimize()
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.25s


Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-0lszl83h has 19488 rows; 8064 cols; 29736 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 2e+02]
  Cost    [1e+00, 1e+04]
  Bound   [0e+00, 0e+00]
  RHS     [7e-03, 5e+02]
Presolving model
2285 rows, 5908 cols, 9304 nonzeros 0s
1447 rows, 4918 cols, 7721 nonzeros 0s
1132 rows, 3482 cols, 5884 nonzeros 0s
Dependent equations search running on 1132 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
1132 rows, 3482 cols, 5884 nonzeros 0s
Presolve reductions: rows 1132(-18356); columns 3482(-4582); nonzeros 5884(-23852) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.1s
       1330    -4.0342267868e+06 Pr: 0(0); Du: 0(3.55271e-15

INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 8064 primals, 19488 duals
Objective: -4.03e+06
Solver: highs
Runtime: 0.18s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper, Kirchhoff-Voltage-Law were not assigned to the network.


('ok', 'optimal')

In [33]:
new_bus = "New Battery Node"
scenario_paths = ("networks/SV2024_north-west.nc", "networks/WP2024_north-west.nc")
chosen = 1
x, y = -8.5, 54.5  # lon, lat (deg)
new_bus_v_nom = 110

# CHANGED: sweep of K values to test, instead of a single hardcoded K = 3
k_values = [1, 2, 3, 4, 5]
results = []  # CHANGED: collects one dict of metrics per K value


def total_curtailment_mwh(network, renewable_carriers=("wind",)):
    gens = network.generators
    curtailable = gens[gens.carrier.isin(renewable_carriers)].index

    available = (
        network.generators_t.p_max_pu[curtailable]
        * gens.loc[curtailable, "p_nom_opt"]
    )
    dispatched = network.generators_t.p[curtailable]

    curtailed_mw = (available - dispatched).clip(lower=0)
    weighted = curtailed_mw.mul(network.snapshot_weightings.generators, axis=0)
    return weighted.sum().sum()


def haversine_km(lon0, lat0, lon1, lat1):
    """Return great-circle distance in kilometres."""
    R = 6371.0
    p0, p1 = np.radians(lat0), np.radians(lat1)
    dphi = np.radians(lat1 - lat0)
    dlambda = np.radians(lon1 - lon0)
    a = np.sin(dphi / 2)**2 + np.cos(p0) * np.cos(p1) * np.sin(dlambda / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


def surplus_waste_reduction_pct(network, curtailment_before, curtailment_after,
                                  renewable_carriers=("wind",)):
    gens = network.generators
    curtailable = gens[gens.carrier.isin(renewable_carriers)].index
    available = (
        network.generators_t.p_max_pu[curtailable]
        * gens.loc[curtailable, "p_nom_opt"]
    )
    total_available_mwh = available.mul(
        network.snapshot_weightings.generators, axis=0
    ).sum().sum()

    avoided_mwh = curtailment_before - curtailment_after
    return avoided_mwh / total_available_mwh * 100


n_baseline = pypsa.Network(scenario_paths[chosen])
n_baseline.optimize()
curtailment_before = total_curtailment_mwh(n_baseline)

n = pypsa.Network(scenario_paths[chosen])
n.snapshot_weightings["objective"] *= 8760 / 168

curtailable = n.generators.index[n.generators.carrier == "wind"]
curtailment_penalty = 100
n.generators.loc[curtailable, "marginal_cost"] -= curtailment_penalty

n.add("Bus", new_bus, x=x, y=y, v_nom=new_bus_v_nom)

n.add("StorageUnit",
      f"Battery at {new_bus}",
      bus=new_bus,
      p_nom_extendable=True,
      p_nom_min=0,
      p_nom_max=2_000,
      capital_cost=75_000,
      marginal_cost=0.1,
      efficiency_store=0.95,
      efficiency_dispatch=0.95,
      max_hours=4,
      cyclic_state_of_charge=True)

line_cost_per_mw_km = 300
x_per_km = 0.35  # ohm/km
r_per_km = 0.12  # ohm/km

candidate_buses = [
    bus for bus in n.buses.index
    if bus != new_bus and n.buses.at[bus, "v_nom"] == new_bus_v_nom
]
skipped = set(n.buses.index) - set(candidate_buses) - {new_bus}
if skipped:
    print(f"Skipping {len(skipped)} buses at a different voltage "
          f"(would need a Transformer, not a Line): {sorted(skipped)[:5]}...")

for bus in candidate_buses:
    x0, y0 = n.buses.at[bus, "x"], n.buses.at[bus, "y"]
    length = haversine_km(x, y, x0, y0)
    n.add("Line",
          f"{new_bus} - {bus}",
          bus0=new_bus, bus1=bus,
          x=x_per_km * length,
          r=r_per_km * length,
          s_nom_extendable=True,
          s_nom_min=0,
          s_nom_max=200,
          capital_cost=length * line_cost_per_mw_km,
          length=length)

# This first-pass optimize only needs to run once — it doesn't depend on K,
# it just ranks candidate buses by how much line capacity gets built to them.
n.optimize()

curtailment_after_lines_only = total_curtailment_mwh(n)
print(f"Curtailment after lines alone: {curtailment_after_lines_only:.2f} MWh "
      f"({(curtailment_before - curtailment_after_lines_only) / curtailment_before * 100:.1f}% reduction)")

built = {}
for line_name in n.lines.index:
    if line_name.startswith(new_bus):
        s_opt = n.lines.at[line_name, "s_nom_opt"]
        if s_opt > 0.01:
            existing_bus = line_name.split(" - ")[1]
            built[existing_bus] = s_opt

ranked_buses = sorted(built, key=built.get, reverse=True)  # CHANGED: no longer sliced to top K here — the full ranking is kept, and each loop iteration slices its own K

# CHANGED: remove the exploratory first-pass lines once, before entering the K loop
first_pass_lines = n.lines.index[n.lines.index.str.startswith(new_bus)]
n.remove("Line", first_pass_lines)

# CHANGED: snapshot the network right after the exploratory lines are stripped out,
# so every K iteration starts from an identical, clean state instead of carrying
# over leftover line/battery values from the previous K's optimize() call.
n.model.solver_model = None
n_clean = n.copy()

min_line_cap = 0

# CHANGED: entire block below is now wrapped in a for loop over k_values,
# using a fresh copy of n_clean each time so results for different K don't
# contaminate each other.
for K in k_values:
    n.model.solver_model = None
    n_k = n_clean.copy()  # CHANGED: fresh network per K
    top_buses = ranked_buses[:K]  # CHANGED: slice to this iteration's K

    for bus in top_buses:
        x0, y0 = n_k.buses.at[bus, "x"], n_k.buses.at[bus, "y"]
        length = haversine_km(x, y, x0, y0)
        n_k.add("Line",
                f"{new_bus} - {bus}",
                bus0=new_bus, bus1=bus,
                x=x_per_km * length,
                r=r_per_km * length,
                s_nom_extendable=True,
                s_nom_min=min_line_cap,
                s_nom_max=200,
                capital_cost=length * line_cost_per_mw_km,
                length=length)

    n_k.optimize()

    final_lines = {}
    for line_name in n_k.lines.index:
        if line_name.startswith(new_bus):
            s_opt = n_k.lines.at[line_name, "s_nom_opt"]
            if s_opt > 0.1:
                final_lines[line_name] = s_opt

    battery_opt = n_k.storage_units.at[f"Battery at {new_bus}", "p_nom_opt"]
    curtailment_after = total_curtailment_mwh(n_k)

    if curtailment_before > 0:
        pct_change = (curtailment_after - curtailment_before) / curtailment_before * 100
    else:
        pct_change = float("nan")

    waste_reduction = surplus_waste_reduction_pct(n_k, curtailment_before, curtailment_after)

    # CHANGED: store this K's results instead of just printing them
    results.append({
        "K": K,
        "top_buses": top_buses,
        "built_lines": final_lines,
        "battery_mw": battery_opt,
        "battery_mwh": battery_opt * 4,
        "curtailment_before_mwh": curtailment_before,
        "curtailment_after_mwh": curtailment_after,
        "dispatch_down_pct_change": pct_change,
        "surplus_waste_reduction_pct": waste_reduction,
        "network": n_k,  # NEW: keep the solved network so we can plot the best one later
    })

# CHANGED: display all K results together at the end, instead of printing inline per-run
print(f"\n{'K':>3} | {'Battery (MW)':>13} | {'Battery (MWh)':>14} | {'Curtailment after (MWh)':>24} | {'Dispatch-down Δ%':>17} | {'Waste reduced %':>16}")
print("-" * 100)
for r in results:
    print(f"{r['K']:>3} | {r['battery_mw']:>13.2f} | {r['battery_mwh']:>14.2f} | "
          f"{r['curtailment_after_mwh']:>24.2f} | {r['dispatch_down_pct_change']:>17.1f} | "
          f"{r['surplus_waste_reduction_pct']:>16.2f}")


# Optimal K = the one that reduces curtailment the most (highest waste-reduction %)
best_result = max(results, key=lambda r: r["surplus_waste_reduction_pct"])
best_K = best_result["K"]
n_best = best_result["network"]

print(f"\nOptimal K = {best_K} "
      f"(waste reduction: {best_result['surplus_waste_reduction_pct']:.2f}%, "
      f"battery: {best_result['battery_mw']:.2f} MW)")


INFO:pypsa.network.io:Imported network 'TYTFS2024_WP2024_V35 north-west (aggregated)' has buses, carriers, generators, lines, loads
/tmp/ipykernel_5347/2144651247.py:54: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n_baseline.optimize()
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.03s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 8064 primals, 19488 duals
Objective: -4.03e+06
Solver: highs
Runtime: 0.05s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper, Kirchhoff-Voltage-Law 

Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-lei85aov has 19488 rows; 8064 cols; 29736 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 2e+02]
  Cost    [1e+00, 1e+04]
  Bound   [0e+00, 0e+00]
  RHS     [7e-03, 5e+02]
Presolving model
2285 rows, 5908 cols, 9304 nonzeros 0s
1447 rows, 4918 cols, 7721 nonzeros 0s
1132 rows, 3482 cols, 5884 nonzeros 0s
Dependent equations search running on 1132 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
1132 rows, 3482 cols, 5884 nonzeros 0s
Presolve reductions: rows 1132(-18356); columns 3482(-4582); nonzeros 5884(-23852) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.0s
       1330    -4.0342267868e+06 Pr: 0(0); Du: 0(3.55271e-15

INFO:pypsa.network.io:Imported network 'TYTFS2024_WP2024_V35 north-west (aggregated)' has buses, carriers, generators, lines, loads
/tmp/ipykernel_5347/2144651247.py:108: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n.optimize()


Skipping 1 buses at a different voltage (would need a Transformer, not a Line): ['Srananagh 220']...


INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 7/7 [00:00<00:00, 804.70it/s]
INFO:linopy.io: Writing time: 0.06s


...

Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-w4ry9dqc has 22184 rows; 9076 cols; 36632 nonzeros
Coefficient ranges:
  Matrix  [9e-01, 2e+02]
  Cost    [5e+00, 5e+05]
  Bound   [0e+00, 0e+00]
  RHS     [7e-03, 2e+03]
Presolving model
4698 rows, 7377 cols, 16434 nonzeros 0s
3418 rows, 6097 cols, 16597 nonzeros 0s
Dependent equations search running on 1643 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
3155 rows, 5165 cols, 16415 nonzeros 0s
Presolve reductions: rows 3155(-19029); columns 5165(-3911); nonzeros 16415(-20217) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.0s
       2254    -4.5732033510e+08 Pr: 0(0); Du: 0(1.95317e-10) 0.1s

Performed postsolve
Solving

/tmp/ipykernel_5347/2144651247.py:158: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n_k.optimize()
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.06s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 9245 primals, 22690 duals
Objective: -4.65e+08
Solver: highs
Runtime: 0.08s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper, Line-ext-s-lower, Line-ext-s-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-u

Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-xoe54jmj has 22690 rows; 9245 cols; 38314 nonzeros
Coefficient ranges:
  Matrix  [9e-01, 2e+02]
  Cost    [5e+00, 5e+05]
  Bound   [0e+00, 0e+00]
  RHS     [7e-03, 2e+03]
Presolving model
5202 rows, 7546 cols, 18114 nonzeros 0s
3754 rows, 6098 cols, 18443 nonzeros 0s
Dependent equations search running on 1628 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
3476 rows, 5151 cols, 18650 nonzeros 0s
Presolve reductions: rows 3476(-19214); columns 5151(-4094); nonzeros 18650(-19664) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.0s
       2102    -4.6520937959e+08 Pr: 0(0); Du: 0(2.14741e-10) 0.1s

Performed postsolve
Solving

/tmp/ipykernel_5347/2144651247.py:158: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n_k.optimize()
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.06s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 9414 primals, 23196 duals
Objective: -4.71e+08
Solver: highs
Runtime: 0.11s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper, Line-ext-s-lower, Line-ext-s-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-u

Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-og4mpqt4 has 23196 rows; 9414 cols; 39996 nonzeros
Coefficient ranges:
  Matrix  [9e-01, 2e+02]
  Cost    [5e+00, 5e+05]
  Bound   [0e+00, 0e+00]
  RHS     [7e-03, 2e+03]
Presolving model
5873 rows, 7882 cols, 20127 nonzeros 0s
4258 rows, 6267 cols, 20793 nonzeros 0s
Dependent equations search running on 1920 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
4104 rows, 5611 cols, 21492 nonzeros 0s
Presolve reductions: rows 4104(-19092); columns 5611(-3803); nonzeros 21492(-18504) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.0s
       2535    -4.7137376648e+08 Pr: 0(0); Du: 0(8.84884e-09) 0.1s

Performed postsolve
Solving

In [34]:
# detach the solved model before plotting, in case you want to .copy() n_best later
n_best.model.solver_model = None

bus_colours = pd.Series("lightgray", index=n_best.buses.index, dtype="object")

wind_buses = n_best.generators.loc[n_best.generators.carrier == "wind", "bus"]
other_generator_buses = n_best.generators.loc[n_best.generators.carrier != "wind", "bus"]
load_buses = n_best.loads["bus"]
battery_buses = n_best.storage_units["bus"]

bus_colours.loc[other_generator_buses.unique()] = "red"
bus_colours.loc[wind_buses.unique()] = "green"
bus_colours.loc[load_buses.unique()] = "blue"
bus_colours.loc[battery_buses.unique()] = "orange"

n_best.plot(bus_sizes=0.0025, margin=0.25, bus_colors=bus_colours)
plt.title(f"Optimal network topology (K={best_K})")
plt.show()

qt.qpa.wayland: Wayland does not support QWindow::requestActivate()


In [46]:
print(curtailment_before, curtailment_after)

5602.345625059608 12236.710331002443


In [12]:
print(n.snapshot_weightings.objective.sum())

168.0
